# 07 — Async Programming
**References:** Ramalho *Fluent Python* Ch. 21 · Python docs: asyncio · PEP 492, 525, 530

## Narrative thread
```
Coroutines -> async/await -> Event loop -> Tasks -> gather/wait -> Queues -> Patterns
```

## 1. Coroutines and the event loop

**asyncio** implements **cooperative multitasking** using coroutines. A coroutine voluntarily suspends at `await` points, allowing the event loop to run other coroutines.

**Key concepts:**

| Term | Meaning |
|---|---|
| `async def` | Defines a coroutine function |
| `await expr` | Suspend until `expr` completes |
| `asyncio.Task` | Scheduled coroutine — scheduled on the event loop |
| Event loop | Manages and drives all coroutines |
| `asyncio.run()` | Create an event loop, run a coroutine, close the loop |

**Critical rule:** Never block the event loop with synchronous I/O or `time.sleep()` — use `await asyncio.sleep()` and `asyncio.to_thread()` for blocking work.

In [ ]:
import asyncio
import time
import random
from typing import AsyncIterator

# ── Basic coroutine: compare sequential vs concurrent ─────────────────────
async def fetch_data(name: str, delay: float) -> dict:
    """Simulate async I/O (e.g., HTTP request)."""
    print(f'  [{name}] starting, will take {delay:.1f}s')
    await asyncio.sleep(delay)     # releases the event loop during wait
    print(f'  [{name}] done')
    return {'name': name, 'delay': delay}

async def sequential():
    t0 = time.perf_counter()
    r1 = await fetch_data('A', 0.3)
    r2 = await fetch_data('B', 0.2)
    r3 = await fetch_data('C', 0.1)
    return time.perf_counter() - t0

async def concurrent():
    t0 = time.perf_counter()
    results = await asyncio.gather(
        fetch_data('A', 0.3),
        fetch_data('B', 0.2),
        fetch_data('C', 0.1),
    )
    return time.perf_counter() - t0, results

print('=== Sequential ===')
t_seq = asyncio.run(sequential())
print(f'Total: {t_seq:.3f}s (sum of all delays)')

print()
print('=== Concurrent (asyncio.gather) ===')
t_con, results = asyncio.run(concurrent())
print(f'Total: {t_con:.3f}s (max of all delays)')
print(f'Speedup: {t_seq/t_con:.1f}x')

In [ ]:
# ── asyncio.Task: fire-and-forget with cancellation ───────────────────────
async def worker(name: str, q: asyncio.Queue):
    while True:
        job = await q.get()
        if job is None:             # poison pill pattern
            q.task_done()
            print(f'  [{name}] shutting down')
            return
        print(f'  [{name}] processing job {job}')
        await asyncio.sleep(0.05)   # simulate work
        q.task_done()

async def producer_consumer():
    q = asyncio.Queue(maxsize=5)

    # Start workers
    workers = [
        asyncio.create_task(worker(f'W{i}', q))
        for i in range(3)
    ]

    # Produce jobs
    for i in range(9):
        await q.put(i)

    # Poison pills to shut down workers
    for _ in workers:
        await q.put(None)

    await q.join()   # wait until all items processed
    await asyncio.gather(*workers)

print('=== Producer-Consumer with asyncio.Queue ===')
asyncio.run(producer_consumer())

In [ ]:
# ── asyncio.wait: first-to-finish and timeout patterns ────────────────────
async def race_demo():
    tasks = [asyncio.create_task(fetch_data(f'T{i}', random.uniform(0.1, 0.5)))
             for i in range(5)]

    # Return as soon as FIRST task completes
    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
    winner = done.pop().result()
    print(f'First done: {winner}')

    # Cancel the rest
    for t in pending:
        t.cancel()
    await asyncio.gather(*pending, return_exceptions=True)
    print(f'Cancelled {len(pending)} remaining tasks')

print('=== Race pattern ===')
asyncio.run(race_demo())

print()
# ── asyncio.timeout (Python 3.11+) ────────────────────────────────────────
async def with_timeout():
    try:
        async with asyncio.timeout(0.1):
            await asyncio.sleep(0.5)   # takes longer than timeout
    except TimeoutError:
        print('Task timed out — TimeoutError caught')

asyncio.run(with_timeout())

In [ ]:
# ── Async generators and async iteration ─────────────────────────────────
async def async_range(start: int, stop: int, delay: float = 0.01) -> AsyncIterator[int]:
    for i in range(start, stop):
        await asyncio.sleep(delay)
        yield i

async def demo_async_gen():
    result = []
    async for value in async_range(0, 5, delay=0.005):
        result.append(value)
    # async comprehension
    squared = [v**2 async for v in async_range(0, 5, delay=0.005)]
    print(f'Async iteration:     {result}')
    print(f'Async comprehension: {squared}')

asyncio.run(demo_async_gen())

print()
# ── asyncio.to_thread: run blocking code without blocking event loop ───────
import hashlib

def cpu_intensive(data: bytes) -> str:
    """Blocking CPU work — would block event loop if called directly."""
    return hashlib.sha256(data).hexdigest()

async def demo_to_thread():
    data = b'x' * 10_000_000
    # asyncio.to_thread runs it in a thread pool, releases the event loop
    hash_val = await asyncio.to_thread(cpu_intensive, data)
    print(f'Hash (via to_thread): {hash_val[:16]}...')

asyncio.run(demo_to_thread())

## 2. Key takeaways

| Concept | Rule |
|---|---|
| `async def` | Defines a coroutine; calling it returns a coroutine object, NOT the result |
| `await` | Only valid inside `async def`; yields control back to event loop |
| `asyncio.gather` | Run multiple coroutines concurrently; returns list of results |
| `asyncio.create_task` | Schedule a coroutine immediately; returns `Task` |
| `asyncio.Queue` | Thread-safe-equivalent for async code; use for producer/consumer |
| Never block event loop | Use `await asyncio.sleep()`, `asyncio.to_thread()` for blocking ops |
| `asyncio.timeout` | Python 3.11+ clean timeout pattern |

**Next:** notebook 08 — memory management, profiling, and performance optimization.